In [8]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import numpy as np
import clahe 

In [9]:
training_path = "../Datasets/data/train/Medical/X-Ray/shoulder/Train"
testing_path = "../Datasets/data/test/Medical/X-Ray/shoulder/Test"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/shoulder/valid"

In [10]:
train_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input)
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input)
)

In [11]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 8,
    class_mode = "binary",
    shuffle = False
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 8,
    class_mode = "binary"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 8,
    class_mode = "binary",
    shuffle = False
)

Found 5924 images belonging to 2 classes.
Found 809 images belonging to 2 classes.
Found 1344 images belonging to 2 classes.


In [12]:
training_set.class_indices

{'abnormal': 0, 'normal': 1}

In [13]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256,activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.2)(x)
x = Dense(256)(x)
output = Dense(1,activation="sigmoid")(x)

In [15]:
model = Model(inputs = base_model.input,outputs = output)

In [16]:
model.compile(optimizer=Adam(learning_rate=0.00273),metrics=["accuracy"],loss="binary_crossentropy")

In [17]:
early_stop = EarlyStopping(monitor="val_loss",restore_best_weights=True,patience=4)

In [18]:
model.fit(training_set,validation_data=(valid_set),epochs=40,callbacks=[early_stop])

Epoch 1/40


I0000 00:00:1793688995.337620   36498 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35635__.265


726/741 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.5343 - loss: 0.8558

I0000 00:00:1793689080.481838   36497 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35635__.265


741/741 ━━━━━━━━━━━━━━━━━━━━ 137s 149ms/step - accuracy: 0.5809 - loss: 0.7091 - val_accuracy: 0.6625 - val_loss: 0.6044
Epoch 2/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 80s 107ms/step - accuracy: 0.6713 - loss: 0.6225 - val_accuracy: 0.7058 - val_loss: 0.5650
Epoch 3/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 76s 102ms/step - accuracy: 0.7090 - loss: 0.5750 - val_accuracy: 0.7441 - val_loss: 0.5129
Epoch 4/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 85s 114ms/step - accuracy: 0.7198 - loss: 0.5535 - val_accuracy: 0.7553 - val_loss: 0.5022
Epoch 5/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 82s 110ms/step - accuracy: 0.7215 - loss: 0.5684 - val_accuracy: 0.7528 - val_loss: 0.5053
Epoch 6/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 79s 106ms/step - accuracy: 0.7242 - loss: 0.5481 - val_accuracy: 0.7429 - val_loss: 0.5009
Epoch 7/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 83s 112ms/step - accuracy: 0.7336 - loss: 0.5227 - val_accuracy: 0.7651 - val_loss: 0.4977
Epoch 8/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 82s 111ms/step - accuracy: 0.7382 - loss: 0.5276 - va

In [19]:
model1 = model

In [20]:
prediction = model1.predict(testing_set)


168/168 ━━━━━━━━━━━━━━━━━━━━ 24s 82ms/step


In [21]:
true_prediction = (prediction > 0.57).astype(int)

In [22]:
y_true = testing_set.classes

In [23]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)

              precision    recall  f1-score   support

           0       0.76      0.77      0.76       699
           1       0.75      0.73      0.74       645

    accuracy                           0.75      1344
   macro avg       0.75      0.75      0.75      1344
weighted avg       0.75      0.75      0.75      1344



array([[541, 158],
       [175, 470]])

In [25]:
train_true = training_set.classes 

train_prob = model1.predict(training_set)

train_prediction = (train_prob > 0.57).astype(int)

print(classification_report(train_true, train_prediction))
print(confusion_matrix(train_true, train_prediction))


741/741 ━━━━━━━━━━━━━━━━━━━━ 104s 140ms/step
              precision    recall  f1-score   support

           0       0.80      0.82      0.81      3224
           1       0.78      0.76      0.77      2700

    accuracy                           0.79      5924
   macro avg       0.79      0.79      0.79      5924
weighted avg       0.79      0.79      0.79      5924

[[2638  586]
 [ 660 2040]]


In [ ]:
model1.save('shoulder_x_ray_model.keras')

In [27]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [28]:
model.compile(optimizer=Adam(learning_rate=0.00001),loss="binary_crossentropy",metrics=["accuracy"])

In [29]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [30]:
history = model.fit(training_set,validation_data=valid_set,epochs=40,callbacks=[early_stop, reduce_lr])

Epoch 1/40


I0000 00:00:1793690832.840855   36500 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_150399__.289


517/741 ━━━━━━━━━━━━━━━━━━━━ 18s 82ms/step - accuracy: 0.5589 - loss: 1.4698

I0000 00:00:1793690899.532610   36497 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_150399__.289


741/741 ━━━━━━━━━━━━━━━━━━━━ 131s 125ms/step - accuracy: 0.5652 - loss: 1.1114 - val_accuracy: 0.6860 - val_loss: 0.5799 - learning_rate: 1.0000e-05
Epoch 2/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 65s 87ms/step - accuracy: 0.5836 - loss: 0.7791 - val_accuracy: 0.6613 - val_loss: 0.5838 - learning_rate: 1.0000e-05
Epoch 3/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 67s 90ms/step - accuracy: 0.5920 - loss: 0.7221 - val_accuracy: 0.6934 - val_loss: 0.5472 - learning_rate: 1.0000e-05
Epoch 4/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 57s 77ms/step - accuracy: 0.5977 - loss: 0.6953 - val_accuracy: 0.6761 - val_loss: 0.5589 - learning_rate: 1.0000e-05
Epoch 5/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 62s 83ms/step - accuracy: 0.5998 - loss: 0.6720 - val_accuracy: 0.6972 - val_loss: 0.5435 - learning_rate: 1.0000e-05
Epoch 6/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 64s 86ms/step - accuracy: 0.6057 - loss: 0.6564 - val_accuracy: 0.6947 - val_loss: 0.5437 - learning_rate: 1.0000e-05
Epoch 7/40
741/741 ━━━━━━━━━━━━━━━━━━━━ 64s 86ms/step - accur

In [31]:
result1 = model.predict(testing_set)

168/168 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step


In [32]:
result = (result1 > 0.54).astype(int)

In [33]:
print(classification_report(y_true,result))
print(confusion_matrix(y_true,result))

              precision    recall  f1-score   support

           0       0.81      0.65      0.72       699
           1       0.69      0.84      0.75       645

    accuracy                           0.74      1344
   macro avg       0.75      0.74      0.74      1344
weighted avg       0.75      0.74      0.74      1344

[[453 246]
 [106 539]]


In [ ]:
train_true = training_set.classes 

train_prob = model.predict(training_set)

train_prediction = (train_prob > 0.57).astype(int)

print(classification_report(train_true, train_prediction))
print(confusion_matrix(train_true, train_prediction))


741/741 ━━━━━━━━━━━━━━━━━━━━ 64s 87ms/step
              precision    recall  f1-score   support

           0       0.81      0.76      0.79      3224
           1       0.74      0.79      0.76      2700

    accuracy                           0.78      5924
   macro avg       0.77      0.78      0.77      5924
weighted avg       0.78      0.78      0.78      5924

[[2461  763]
 [ 566 2134]]


: 